# 🚀 SAIR PyTorch Mastery - Lecture 6A: Autoencoders & GANs
## From Compression to Creation

**Course:** Ultimate Applied Deep Learning with PyTorch  
**Module:** Generative AI Foundations  
**Instructor:** Mohammed Awad Ahmed (Silva)  
**SAIR Community:** Building Sudan's AI Future 🇸🇩

---

## 📘 How to Use This Notebook

This is a **standalone, self-teaching lecture notebook** designed to replace traditional video lectures. Here's how to get the most from it:

**How to Study:**
1. **Read all markdown cells carefully** - they contain explanations, mental models, and reasoning
2. **Run code cells sequentially** - don't skip ahead as each builds on the previous
3. **Pause at "Stop & Think" prompts** - make predictions before running the next cell
4. **Play with the interactive widgets** - move sliders, see real-time changes
5. **Experiment with the code** - change parameters and see what happens

**Time Commitment:** ~3-4 hours for complete understanding

**Learning Outcomes:** After completing this notebook, you will be able to:
1. Explain how autoencoders learn efficient data representations
2. Build autoencoders from scratch for dimensionality reduction
3. Use autoencoders for anomaly detection in real-world scenarios
4. Implement Variational Autoencoders (VAEs) and understand the math
5. Generate new images with VAEs and GANs
6. Diagnose and fix GAN training issues like mode collapse
7. Apply unsupervised pretraining when you have limited labeled data

**This Notebook Builds On Lectures 3-5:**
- Lecture 3: CNNs and feature extraction
- Lecture 4: Transfer learning concepts
- Lecture 5: Detection architectures
- This Lecture: Generative models

## 📈 YOUR LEARNING JOURNEY TRACKER

| Part | Section | Status | Time | Confidence |
|------|---------|--------|------|------------|
| **Part 0** | Quick Win: Generate in 5 Lines | □ | __ min | __ |
| **Part 1** | The Memory Experiment | □ | __ min | __ |
| **Part 2** | Undercomplete Autoencoders = PCA | □ | __ min | __ |
| **Part 3** | Stacked Autoencoders from Scratch | □ | __ min | __ |
| **Part 4** | Anomaly Detection in Action | □ | __ min | __ |
| **Part 5** | Visualizing Latent Space | □ | __ min | __ |
| **Part 6** | Denoising & Sparse Autoencoders | □ | __ min | __ |
| **Part 7** | Variational Autoencoders (The Math) | □ | __ min | __ |
| **🎥 DEMO 1** | VAE Generates Fashion MNIST | □ | __ min | __ |
| **Part 8** | Semantic Interpolation | □ | __ min | __ |
| **Part 9** | GANs: The Counterfeiters | □ | __ min | __ |
| **🎥 DEMO 2** | GAN Training & Mode Collapse | □ | __ min | __ |
| **🎯 MAIN EVENT** | Unsupervised Pretraining | □ | __ min | __ |
| **🏆 Portfolio** | 8 Projects to Build | □ | __ min | __ |

**Goal:** Complete all parts with confidence ≥4/5  
**Expected Total Time:** 3-4 hours

## 🛠️ Setup & Environment

In [ ]:
# Initial Setup and Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import time
import warnings
from collections import namedtuple
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.manifold import TSNE
import gc

warnings.filterwarnings('ignore')

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load Fashion MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

print(f"\n📊 Fashion MNIST Dataset:")
print(f"   Training samples: {len(train_data)}")
print(f"   Test samples: {len(test_data)}")
print(f"   Image shape: {train_data[0][0].shape}")
print(f"   Classes: {train_data.classes}")

# Show sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    img, label = train_data[i]
    ax = axes[i//5, i%5]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(train_data.classes[label])
    ax.axis('off')
plt.suptitle("Fashion MNIST Samples", fontsize=14)
plt.tight_layout()
plt.show()

# 🚀 PART 0: Quick Win - Generate in 5 Lines

## Before we build anything, let's see what's possible

**We'll use a pre-trained VAE to generate fashion items instantly.**

Don't worry about the code yet - this is just to show you where we're going!

In [ ]:
print("="*60)
print("🚀 QUICK WIN: Generate Fashion Items in 5 Lines")
print("="*60)

# We'll use a pre-trained VAE (simplified version)
# In 5 lines, we can generate new fashion items!

# This is a placeholder - we'll build this model from scratch in Part 7
# But here's what it will look like:

print("""
```python
# Load pre-trained VAE (what we'll build today)
vae = load_pretrained_vae('fashion-mnist')

# Generate random latent codes
z = torch.randn(16, 32)  # 16 random codes

# Generate images!
generated = vae.decode(z)

# Show them
show_images(generated)
```
""")

print("\n🎯 By the end of this lecture, YOU will build this from scratch!")
print("   First, we need to understand HOW it works...")

# 🧠 PART 1: The Memory Experiment

## Why Constraints Make Us Smarter

**Which sequence is easier to memorize?**

- Sequence A: 40, 27, 25, 36, 81, 57, 10, 73, 19, 68
- Sequence B: 50, 48, 46, 44, 42, 40, 38, 36, 34, 32, 30, 28, 26, 24, 22, 20, 18, 16, 14

At first glance, Sequence A seems easier (it's shorter). But look closer at Sequence B...

> "Chess experts don't have better memory—they just see patterns."  
> — Chase & Simon, 1973

In [ ]:
print("="*60)
print("🧠 THE MEMORY EXPERIMENT")
print("="*60)

seq_a = [40, 27, 25, 36, 81, 57, 10, 73, 19, 68]
seq_b = [50, 48, 46, 44, 42, 40, 38, 36, 34, 32, 30, 28, 26, 24, 22, 20, 18, 16, 14]

@widgets.interact(
    show_pattern=widgets.Checkbox(value=False, description='Show pattern')
)
def memory_demo(show_pattern):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Sequence A
    axes[0].bar(range(len(seq_a)), seq_a, color='skyblue')
    axes[0].set_title(f'Sequence A: {len(seq_a)} numbers')
    axes[0].set_xlabel('Position')
    axes[0].set_ylabel('Value')
    
    # Sequence B
    axes[1].bar(range(len(seq_b)), seq_b, color='lightcoral')
    axes[1].set_title(f'Sequence B: {len(seq_b)} numbers')
    axes[1].set_xlabel('Position')
    axes[1].set_ylabel('Value')
    
    if show_pattern:
        axes[1].axhline(y=50, color='red', linestyle='--', alpha=0.5)
        axes[1].axhline(y=14, color='red', linestyle='--', alpha=0.5)
        axes[1].text(15, 52, 'Start: 50', fontsize=10, color='red')
        axes[1].text(15, 16, 'End: 14', fontsize=10, color='red')
        axes[1].text(5, 30, 'Even numbers decreasing by 2', 
                     fontsize=12, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))
    
    plt.tight_layout()
    plt.show()
    
    if show_pattern:
        print("\n🎯 PATTERN DETECTED: Even numbers from 50 down to 14")
        print("   Now Sequence B is EASIER to remember than Sequence A!")
    else:
        print("\n🤔 Which is easier to memorize? Try checking 'Show pattern'...")

print("\n💡 KEY INSIGHT: Constraints force us to find patterns!")
print("   • Without pattern: Must memorize each number individually")
print("   • With pattern: Just remember 'even numbers 50→14'")
print("\n🎯 This is EXACTLY how autoencoders work!")

## 🔗 Connecting to Autoencoders

Just like the memory experiment, autoencoders:
1. **Take in data** (like the long sequence)
2. **Are constrained** (forced to find patterns)
3. **Learn efficient representations** (like "even numbers 50→14")
4. **Can reconstruct** (regenerate the full sequence)

**The constraint is what makes them learn!**

# 📐 PART 2: Undercomplete Autoencoders = PCA

## The Simplest Autoencoder: Linear + MSE = PCA

Let's build the most basic autoencoder: a linear layer that compresses 3D data to 2D.

In [ ]:
print("="*60)
print("📐 UNDERCOMPLETE LINEAR AUTOENCODER = PCA")
print("="*60)

# Generate 3D dataset (like a flattened helix)
def generate_3d_data(n_samples=1000):
    t = np.linspace(0, 4*np.pi, n_samples)
    x = np.sin(t) * t
    y = np.cos(t) * t
    z = t
    X = np.column_stack([x, y, z])
    return torch.tensor(X, dtype=torch.float32)

X_train = generate_3d_data()
print(f"Generated 3D dataset shape: {X_train.shape}")

# Create linear autoencoder
encoder = nn.Linear(3, 2)
decoder = nn.Linear(2, 3)
autoencoder = nn.Sequential(encoder, decoder)

# Training setup
train_set = TensorDataset(X_train, X_train)  # Input = target
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
optimizer = optim.Adam(autoencoder.parameters(), lr=0.01)
criterion = nn.MSELoss()

# Train
epochs = 50
losses = []
for epoch in range(epochs):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        output = autoencoder(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    losses.append(total_loss / len(train_loader))
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {losses[-1]:.4f}")

# Visualize
with torch.no_grad():
    codings = encoder(X_train).numpy()
    reconstructions = autoencoder(X_train).numpy()

fig = plt.figure(figsize=(15, 5))

# Original 3D
ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(X_train[:, 0], X_train[:, 1], X_train[:, 2], c='blue', alpha=0.6)
ax1.set_title('Original 3D Data')

# 2D latent space
ax2 = fig.add_subplot(132)
ax2.scatter(codings[:, 0], codings[:, 1], c=range(len(codings)), cmap='viridis', alpha=0.6)
ax2.set_title('2D Latent Space (Compressed)')
ax2.set_xlabel('Dimension 1')
ax2.set_ylabel('Dimension 2')
ax2.grid(True, alpha=0.3)

# Reconstructed 3D
ax3 = fig.add_subplot(133, projection='3d')
ax3.scatter(reconstructions[:, 0], reconstructions[:, 1], reconstructions[:, 2], 
           c='red', alpha=0.6)
ax3.set_title('Reconstructed 3D Data')

plt.tight_layout()
plt.show()

print("\n🎯 KEY INSIGHT: This is EXACTLY what PCA does!")
print("   The autoencoder found the best 2D plane to project the data onto,")
print("   preserving as much variance as possible.")

## 🎮 Interactive: Explore the Latent Space

Move the sliders to see how different 2D points map back to 3D!

In [ ]:
@widgets.interact(
    dim1=widgets.FloatSlider(min=-3, max=3, step=0.1, value=0, description='Dim 1:'),
    dim2=widgets.FloatSlider(min=-3, max=3, step=0.1, value=0, description='Dim 2:')
)
def explore_latent(dim1, dim2):
    with torch.no_grad():
        # Create a point in latent space
        z = torch.tensor([[dim1, dim2]], dtype=torch.float32)
        # Decode to 3D
        x_recon = decoder(z).numpy()[0]
        
    fig = plt.figure(figsize=(10, 4))
    
    # Latent space
    ax1 = fig.add_subplot(121)
    ax1.scatter(codings[:, 0], codings[:, 1], c='blue', alpha=0.3, label='Training codings')
    ax1.scatter([dim1], [dim2], c='red', s=200, marker='*', label='Your point')
    ax1.set_title('Latent Space')
    ax1.set_xlabel('Dim 1')
    ax1.set_ylabel('Dim 2')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Reconstructed point
    ax2 = fig.add_subplot(122, projection='3d')
    ax2.scatter(X_train[:, 0], X_train[:, 1], X_train[:, 2], 
               c='blue', alpha=0.2, label='Training data')
    ax2.scatter([x_recon[0]], [x_recon[1]], [x_recon[2]], 
               c='red', s=200, marker='*', label='Reconstruction')
    ax2.set_title('Reconstructed Point')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"📍 Latent point: ({dim1:.2f}, {dim2:.2f})")
    print(f"📐 Reconstructed 3D: ({x_recon[0]:.2f}, {x_recon[1]:.2f}, {x_recon[2]:.2f})")

# 🏗️ PART 3: Stacked Autoencoders from Scratch

## Now let's build a deep autoencoder for Fashion MNIST

In [ ]:
print("="*60)
print("🏗️ STACKED AUTOENCODER FOR FASHION MNIST")
print("="*60)

class StackedAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: 784 → 128 → 32
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU()
        )
        # Decoder: 32 → 128 → 784
        self.decoder = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 28*28),
            nn.Sigmoid(),  # Output pixel values in [0,1]
            nn.Unflatten(1, (1, 28, 28))
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

stacked_ae = StackedAutoencoder()
print(f"✅ Model created")
print(f"   Total parameters: {sum(p.numel() for p in stacked_ae.parameters()):,}")

# Training setup
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
optimizer = optim.Adam(stacked_ae.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Quick training (just 5 epochs for demo)
print("\n🚀 Training (5 epochs for demo)...")
for epoch in range(5):
    total_loss = 0
    for images, _ in train_loader:
        optimizer.zero_grad()
        outputs = stacked_ae(images)
        loss = criterion(outputs, images)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/5, Loss: {total_loss/len(train_loader):.4f}")

## 👁️ Visualizing Reconstructions

Let's see how well our autoencoder reconstructs images.

In [ ]:
def show_reconstructions(model, dataset, n=10):
    model.eval()
    fig, axes = plt.subplots(2, n, figsize=(15, 4))
    
    with torch.no_grad():
        for i in range(n):
            img, label = dataset[i]
            # Add batch dimension
            img_batch = img.unsqueeze(0)
            # Reconstruct
            recon = model(img_batch)
            
            # Original
            axes[0, i].imshow(img.squeeze(), cmap='gray')
            axes[0, i].axis('off')
            if i == 0:
                axes[0, i].set_title('Original', fontsize=12)
            
            # Reconstruction
            axes[1, i].imshow(recon.squeeze(), cmap='gray')
            axes[1, i].axis('off')
            if i == 0:
                axes[1, i].set_title('Reconstructed', fontsize=12)
    
    plt.suptitle('Autoencoder Reconstructions', fontsize=14)
    plt.tight_layout()
    plt.show()

show_reconstructions(stacked_ae, test_data)

print("\n🧠 Notice: The reconstructions are a bit blurry but recognizable!")
print("   The autoencoder learned to compress 784 pixels → 32 numbers → back to 784")

# 🔍 PART 4: Anomaly Detection in Action

## Real-World Application: Find What Doesn't Belong

**Scenario:** You've trained on Fashion MNIST (clothing). What happens when you show it MNIST digits?

In [ ]:
print("="*60)
print("🔍 ANOMALY DETECTION")
print("="*60)

# Load MNIST (anomalies)
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

def compute_reconstruction_loss(model, dataset, n_samples=100):
    model.eval()
    losses = []
    with torch.no_grad():
        for i in range(min(n_samples, len(dataset))):
            img, _ = dataset[i]
            img_batch = img.unsqueeze(0)
            recon = model(img_batch)
            loss = F.mse_loss(recon, img_batch).item()
            losses.append(loss)
    return losses

# Compute losses
fashion_losses = compute_reconstruction_loss(stacked_ae, test_data, 200)
mnist_losses = compute_reconstruction_loss(stacked_ae, mnist_test, 200)

# Visualize
@widgets.interact(
    threshold=widgets.FloatSlider(min=0.02, max=0.15, step=0.005, value=0.06, 
                                   description='Threshold:')
)
def anomaly_detector(threshold):
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    
    # Loss distributions
    ax1 = axes[0, 0]
    ax1.hist(fashion_losses, bins=30, alpha=0.7, label='Fashion MNIST', color='blue')
    ax1.hist(mnist_losses, bins=30, alpha=0.7, label='MNIST (anomalies)', color='red')
    ax1.axvline(threshold, color='green', linestyle='--', linewidth=2, label='Threshold')
    ax1.set_xlabel('Reconstruction Loss')
    ax1.set_ylabel('Count')
    ax1.set_title('Loss Distribution')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Show examples
    with torch.no_grad():
        # Normal examples (Fashion)
        for i in range(3):
            img, label = test_data[i]
            img_batch = img.unsqueeze(0)
            recon = stacked_ae(img_batch)
            loss = F.mse_loss(recon, img_batch).item()
            
            axes[1, i].imshow(recon.squeeze(), cmap='gray')
            color = 'green' if loss < threshold else 'red'
            axes[1, i].set_title(f'Fashion: {loss:.4f}', color=color)
            axes[1, i].axis('off')
        
        # Anomalies (MNIST digits)
        for i in range(3):
            img, label = mnist_test[i]
            img_batch = img.unsqueeze(0)
            recon = stacked_ae(img_batch)
            loss = F.mse_loss(recon, img_batch).item()
            
            axes[0, 1+i].imshow(recon.squeeze(), cmap='gray')
            color = 'green' if loss < threshold else 'red'
            axes[0, 1+i].set_title(f'MNIST: {loss:.4f}', color=color)
            axes[0, 1+i].axis('off')
    
    axes[0, 1].set_title('Reconstructions of MNIST (Anomalies)', fontsize=12)
    axes[1, 0].set_title('Reconstructions of Fashion MNIST', fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate accuracy
    fashion_pred = [1 if l < threshold else 0 for l in fashion_losses]
    mnist_pred = [1 if l >= threshold else 0 for l in mnist_losses]
    accuracy = (sum(fashion_pred) + sum(mnist_pred)) / (len(fashion_losses) + len(mnist_losses))
    
    print(f"🎯 At threshold {threshold:.3f}:")
    print(f"   • {sum(fashion_pred)}/{len(fashion_losses)} Fashion images correctly identified as normal")
    print(f"   • {sum(mnist_pred)}/{len(mnist_losses)} MNIST images correctly identified as anomalies")
    print(f"   • Overall accuracy: {accuracy*100:.1f}%")

# 🌌 PART 5: Visualizing Latent Space

## What does the autoencoder "think" about? Let's use t-SNE to see.

In [ ]:
print("="*60)
print("🌌 VISUALIZING LATENT SPACE WITH t-SNE")
print("="*60)

# Get encodings for test set
stacked_ae.eval()
encodings = []
labels = []

with torch.no_grad():
    for i in range(500):  # Use 500 samples for speed
        img, label = test_data[i]
        img_batch = img.unsqueeze(0)
        encoding = stacked_ae.encoder(img_batch).numpy().flatten()
        encodings.append(encoding)
        labels.append(label)

encodings = np.array(encodings)
labels = np.array(labels)

print(f"Encodings shape: {encodings.shape}")

# Apply t-SNE
print("\n🔄 Running t-SNE (this may take a moment)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
encodings_2d = tsne.fit_transform(encodings)

# Visualize
plt.figure(figsize=(12, 8))
scatter = plt.scatter(encodings_2d[:, 0], encodings_2d[:, 1], 
                      c=labels, cmap='tab10', alpha=0.7, s=50)
plt.colorbar(scatter, ticks=range(10), label='Class')
plt.title('t-SNE Visualization of Fashion MNIST Latent Space', fontsize=14)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(True, alpha=0.3)
plt.show()

print("\n🧠 Notice how similar items cluster together!")
print("   • T-shirts/tops (0) and shirts (6) are close")
print("   • Pullovers (2), dresses (3), coats (4) form a cluster")
print("   • Sandals (5), sneakers (7), bags (8), boots (9) are separate")

# 🧹 PART 6: Denoising & Sparse Autoencoders

## 6.1 Denoising Autoencoder: Clean the Noise

In [ ]:
print("="*60)
print("🧹 DENOISING AUTOENCODER")
print("="*60)

class DenoisingAutoencoder(nn.Module):
    def __init__(self, dropout_prob=0.3):
        super().__init__()
        self.dropout = nn.Dropout(dropout_prob)
        # Encoder
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU()
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 28*28),
            nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28))
        )
    
    def forward(self, x):
        # Add noise during training only
        if self.training:
            x_noisy = self.dropout(x.view(x.size(0), -1)).view(x.shape)
        else:
            x_noisy = x
        encoded = self.encoder(x_noisy)
        decoded = self.decoder(encoded)
        return decoded, x_noisy

denoising_ae = DenoisingAutoencoder()

# Quick demo of denoising
denoising_ae.eval()
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

with torch.no_grad():
    for i in range(5):
        img, _ = test_data[i]
        img_batch = img.unsqueeze(0)
        
        # Add noise manually for demo
        noise = torch.randn_like(img_batch) * 0.3
        img_noisy = torch.clamp(img_batch + noise, 0, 1)
        
        # Denoise
        recon, _ = denoising_ae(img_noisy)
        
        # Show noisy
        axes[0, i].imshow(img_noisy.squeeze(), cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_title('Noisy Input', fontsize=12)
        
        # Show denoised
        axes[1, i].imshow(recon.squeeze(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_title('Denoised Output', fontsize=12)

plt.suptitle('Denoising Autoencoder (with dropout noise)', fontsize=14)
plt.tight_layout()
plt.show()

print("\n🎯 This is how real-world denoising works!")
print("   Applications: Old photo restoration, medical image enhancement")

## 6.2 Sparse Autoencoder with KL Divergence

In [ ]:
print("="*60)
print("🔢 SPARSE AUTOENCODER WITH KL DIVERGENCE")
print("="*60)

AEOutput = namedtuple("AEOutput", ["output", "codings"])

class SparseAutoencoder(nn.Module):
    def __init__(self, coding_dim=256, target_sparsity=0.1):
        super().__init__()
        self.target_sparsity = target_sparsity
        
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, coding_dim),
            nn.Sigmoid()  # Constrain to [0,1] for sparsity
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(coding_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 28*28),
            nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28))
        )
    
    def forward(self, x):
        codings = self.encoder(x)
        output = self.decoder(codings)
        return AEOutput(output, codings)

def sparse_loss(y_pred, y_target, target_sparsity=0.1, kl_weight=1e-3, eps=1e-8):
    # Reconstruction loss
    recon_loss = F.mse_loss(y_pred.output, y_target)
    
    # Sparsity loss (KL divergence)
    p = torch.tensor(target_sparsity, device=y_pred.codings.device)
    q = torch.clamp(y_pred.codings.mean(dim=0), eps, 1 - eps)
    kl_div = p * torch.log(p / q) + (1 - p) * torch.log((1 - p) / (1 - q))
    
    return recon_loss + kl_weight * kl_div.sum()

print("✅ Sparse autoencoder with KL divergence defined")
print("\n📐 KL Divergence formula:")
print("   D_KL(p || q) = p * log(p/q) + (1-p) * log((1-p)/(1-q))")

# 🎲 PART 7: Variational Autoencoders (The Math)

## From Deterministic to Probabilistic

**Key Insight:** Instead of learning a single point in latent space, learn a DISTRIBUTION!

In [ ]:
print("="*60)
print("🎲 VARIATIONAL AUTOENCODER")
print("="*60)

VAEOutput = namedtuple("VAEOutput", ["output", "mu", "logvar"])

class VAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Encoder: outputs mu and logvar
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 2 * latent_dim)  # 2x for mu and logvar
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 28*28),
            nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28))
        )
    
    def encode(self, x):
        h = self.encoder(x)
        mu, logvar = h.chunk(2, dim=-1)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """The magic trick that makes backprop possible"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return VAEOutput(recon, mu, logvar)

def vae_loss(y_pred, y_target):
    recon_loss = F.mse_loss(y_pred.output, y_target, reduction='sum')
    
    # KL divergence: -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    kl_loss = -0.5 * torch.sum(1 + y_pred.logvar - y_pred.mu.pow(2) - y_pred.logvar.exp())
    
    return (recon_loss + kl_loss) / y_target.size(0)

vae = VAE(latent_dim=32)
print(f"✅ VAE created with {sum(p.numel() for p in vae.parameters()):,} parameters")

print("\n📐 VAE Loss:")
print("   L = MSE_reconstruction + KL_divergence")
print("   KL = -0.5 * Σ(1 + log(σ²) - μ² - σ²)")

## 🎥 DEMO 1: VAE Generates Fashion MNIST

In [ ]:
print("="*60)
print("🎥 DEMO 1: VAE GENERATES FASHION MNIST")
print("="*60)

# Train VAE (simplified - just 3 epochs for demo)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
optimizer = optim.Adam(vae.parameters(), lr=0.001)

print("🚀 Training VAE (3 epochs for demo)...")
for epoch in range(3):
    total_loss = 0
    for images, _ in train_loader:
        optimizer.zero_grad()
        outputs = vae(images)
        loss = vae_loss(outputs, images)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/3, Loss: {total_loss/len(train_loader):.2f}")

# Generate new images
vae.eval()
with torch.no_grad():
    # Sample random latent vectors
    z = torch.randn(5*5, vae.latent_dim)
    generated = vae.decode(z)

fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for i in range(5):
    for j in range(5):
        idx = i*5 + j
        axes[i, j].imshow(generated[idx].squeeze(), cmap='gray')
        axes[i, j].axis('off')
plt.suptitle('VAE Generated Fashion Items', fontsize=14)
plt.tight_layout()
plt.show()

print("\n🎯 These images never existed before!")
print("   The VAE learned the distribution of fashion items and sampled new ones.")

## 🎮 Interactive: Semantic Interpolation

In [ ]:
@widgets.interact(
    alpha=widgets.FloatSlider(min=0, max=1, step=0.05, value=0.5, description='Interpolation:')
)
def semantic_interpolation(alpha):
    vae.eval()
    with torch.no_grad():
        # Get two real images
        img1, label1 = test_data[0]  # Ankle boot
        img2, label2 = test_data[2]  # Pullover
        
        # Encode them
        mu1, _ = vae.encode(img1.unsqueeze(0))
        mu2, _ = vae.encode(img2.unsqueeze(0))
        
        # Interpolate in latent space
        z_interp = (1 - alpha) * mu1 + alpha * mu2
        
        # Decode
        img_interp = vae.decode(z_interp)
    
    fig, axes = plt.subplots(1, 3, figsize=(10, 4))
    
    axes[0].imshow(img1.squeeze(), cmap='gray')
    axes[0].set_title(f'Start: {test_data.classes[label1]}')
    axes[0].axis('off')
    
    axes[1].imshow(img_interp.squeeze(), cmap='gray')
    axes[1].set_title(f'Interpolation (α={alpha:.2f})')
    axes[1].axis('off')
    
    axes[2].imshow(img2.squeeze(), cmap='gray')
    axes[2].set_title(f'End: {test_data.classes[label2]}')
    axes[2].axis('off')
    
    plt.suptitle('Semantic Interpolation in Latent Space', fontsize=14)
    plt.tight_layout()
    plt.show()
    
print("\n🧠 Notice: The interpolation creates a SMOOTH transition,")
print("   not just a blend of pixels! This is semantic understanding.")

# 🤼 PART 9: GANs - The Counterfeiters

## Generator vs Discriminator: A Zero-Sum Game

> "The generator is like a criminal making counterfeit money,  
>   the discriminator is like the police investigator."  
> — Ian Goodfellow, 2014

In [ ]:
print("="*60)
print("🤼 GENERATIVE ADVERSARIAL NETWORKS")
print("="*60)

latent_dim = 32

# Generator: random noise → fake images
generator = nn.Sequential(
    nn.Linear(latent_dim, 128),
    nn.ReLU(),
    nn.Linear(128, 256),
    nn.ReLU(),
    nn.Linear(256, 28*28),
    nn.Sigmoid(),
    nn.Unflatten(1, (1, 28, 28))
)

# Discriminator: image → real/fake
discriminator = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 1),
    nn.Sigmoid()
)

print(f"✅ Generator: {sum(p.numel() for p in generator.parameters()):,} params")
print(f"✅ Discriminator: {sum(p.numel() for p in discriminator.parameters()):,} params")

def train_gan(generator, discriminator, train_loader, epochs=5):
    criterion = nn.BCELoss()
    g_optimizer = optim.Adam(generator.parameters(), lr=0.0002)
    d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002)
    
    g_losses, d_losses = [], []
    
    for epoch in range(epochs):
        for batch_idx, (real_images, _) in enumerate(train_loader):
            batch_size = real_images.size(0)
            
            # Train Discriminator
            # Real images
            real_labels = torch.ones(batch_size, 1)
            fake_labels = torch.zeros(batch_size, 1)
            
            d_optimizer.zero_grad()
            real_output = discriminator(real_images)
            d_real_loss = criterion(real_output, real_labels)
            
            # Fake images
            z = torch.randn(batch_size, latent_dim)
            fake_images = generator(z)
            fake_output = discriminator(fake_images.detach())
            d_fake_loss = criterion(fake_output, fake_labels)
            
            d_loss = d_real_loss + d_fake_loss
            d_loss.backward()
            d_optimizer.step()
            
            # Train Generator
            g_optimizer.zero_grad()
            z = torch.randn(batch_size, latent_dim)
            fake_images = generator(z)
            fake_output = discriminator(fake_images)
            g_loss = criterion(fake_output, real_labels)  # Want discriminator to think fakes are real
            g_loss.backward()
            g_optimizer.step()
            
            if batch_idx % 100 == 0:
                print(f'Epoch {epoch+1}/{epochs}, Batch {batch_idx}, D Loss: {d_loss.item():.4f}, G Loss: {g_loss.item():.4f}')
        
        g_losses.append(g_loss.item())
        d_losses.append(d_loss.item())
    
    return g_losses, d_losses

print("\n🚀 Training GAN (2 epochs for demo)...")
g_losses, d_losses = train_gan(generator, discriminator, train_loader, epochs=2)

## 🎥 DEMO 2: GAN Training & Mode Collapse

In [ ]:
print("="*60)
print("🎥 DEMO 2: GAN GENERATIONS & MODE COLLAPSE")
print("="*60)

# Generate images with trained GAN
generator.eval()
with torch.no_grad():
    z = torch.randn(5*5, latent_dim)
    fake_images = generator(z)

fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for i in range(5):
    for j in range(5):
        idx = i*5 + j
        axes[i, j].imshow(fake_images[idx].squeeze(), cmap='gray')
        axes[i, j].axis('off')
plt.suptitle('GAN Generated Fashion Items', fontsize=14)
plt.tight_layout()
plt.show()

print("\n⚠️ The Problem: Mode Collapse")
print("   • GANs often generate only 1-2 types of items")
print("   • The generator finds an 'easy' mode and sticks to it")
print("   • This is why diffusion models are now preferred")

# 🎯 THE MAIN EVENT: Unsupervised Pretraining

## Real-World Scenario: 50,000 unlabeled, 500 labeled images

In [ ]:
print("="*60)
print("🎯 THE MAIN EVENT: UNSUPERVISED PRETRAINING")
print("="*60)

print("""
📊 SCENARIO:
   • You have 50,000 unlabeled fashion images (easy to collect)
   • Only 500 labeled images (expensive/hard to get)
   • Goal: Build a classifier for 10 clothing types

💡 SOLUTION:
   1. Train autoencoder on ALL 50,000 unlabeled images
   2. Reuse encoder layers for classifier
   3. Train classifier on just 500 labeled images
""")

# 1. Train autoencoder on all data (we already did this!)
pretrained_encoder = stacked_ae.encoder

# 2. Build classifier reusing encoder
class PretrainedClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        # Freeze encoder? (optional)
        for param in self.encoder.parameters():
            param.requires_grad = False
        self.classifier = nn.Linear(32, 10)
    
    def forward(self, x):
        features = self.encoder(x)
        return self.classifier(features)

# 3. Create small labeled dataset (500 samples)
indices = np.random.choice(len(train_data), 500, replace=False)
small_train = torch.utils.data.Subset(train_data, indices)
small_loader = DataLoader(small_train, batch_size=32, shuffle=True)

print("\n✅ Created classifier with pretrained encoder")
print("   • Encoder frozen (features already learned)")
print("   • Only training the final classification layer")

# 4. Train classifier
pretrained_model = PretrainedClassifier(pretrained_encoder)
optimizer = optim.Adam(pretrained_model.classifier.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

print("\n🚀 Training classifier on 500 samples...")
for epoch in range(5):
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in small_loader:
        optimizer.zero_grad()
        outputs = pretrained_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    print(f"Epoch {epoch+1}/5, Loss: {total_loss/len(small_loader):.4f}, Acc: {100*correct/total:.2f}%")

print("\n🎯 Without pretraining, accuracy would be <50% with only 500 samples!")
print("   This is the power of unsupervised pretraining.")

# 🏆 Portfolio: 8 Projects to Build

Now that you understand the theory, here are real projects you can build:

In [ ]:
projects = """
┌─────────────────────────────────────────────────────────────────┐
│                     8 PORTFOLIO PROJECTS                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│ 1. 🏭 Manufacturing Defect Detector                            │
│    • Train autoencoder on images of good products             │
│    • Products with high reconstruction loss = defects         │
│    • Save millions in quality control!                        │
│                                                                 │
│ 2. 🎨 Style Transfer with VAE                                 │
│    • Encode two images, interpolate in latent space          │
│    • Create smooth morphs between styles                      │
│    • Generate new artistic creations                          │
│                                                                 │
│ 3. 🕵️ Fake Image Detector                                     │
│    • Train GAN, then use discriminator as detector            │
│    • Identify AI-generated images                             │
│    • Critical for fighting misinformation                     │
│                                                                 │
│ 4. 📦 Data Augmentation Engine                                │
│    • Use VAE to generate variations of training images        │
│    • Boost classifier performance with synthetic data         │
│    • Perfect when you have limited data                       │
│                                                                 │
│ 5. 🧬 Medical Image Denoising                                 │
│    • Train denoising autoencoder on clean X-rays              │
│    • Remove noise from low-dose scans                         │
│    • Reduce radiation exposure while maintaining quality      │
│                                                                 │
│ 6. 🎭 Face Generator                                          │
│    • VAE or GAN on celebrity faces                            │
│    • Generate new faces that don't exist                      │
│    • Understand latent space of facial features               │
│                                                                 │
│ 7. 🔍 Feature Extractor for Few-Shot Learning                │
│    • Train autoencoder on unlabeled data (THE MAIN EVENT!)    │
│    • Reuse encoder for classification with few labels         │
│    • State-of-the-art approach for real-world problems        │
│                                                                 │
│ 8. 🎮 Game Asset Generator                                    │
│    • Train GAN on game sprites                                │
│    • Generate infinite variations                             │
│    • Create unique characters, items, environments            │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
"""

print(projects)
print("\n🎯 Pick one and build it this week! Share in #SAIR with #MyFirstGenerativeModel")

# ✅ MASTERY CHECKLIST

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════╗
║                      MASTERY CHECKLIST                            ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║ Before moving to Lecture 6B (Diffusion Models), ensure you can:  ║
║                                                                   ║
║ □ Explain the memory experiment and its connection to autoencoders║
║ □ Build a linear autoencoder that performs PCA                   ║
║ □ Implement a stacked autoencoder from scratch                   ║
║ □ Use autoencoders for anomaly detection                          ║
║ □ Visualize latent space with t-SNE                              ║
║ □ Understand the reparameterization trick in VAEs                ║
║ □ Generate new images with a trained VAE                         ║
║ □ Perform semantic interpolation in latent space                 ║
║ □ Explain the GAN generator/discriminator game                   ║
║ □ Identify mode collapse in GAN training                         ║
║ □ Apply unsupervised pretraining with limited labeled data       ║
║                                                                   ║
║ 🎉 If you checked all: YOU'RE READY FOR LECTURE 6B!              ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
""")

# 📝 EXERCISES

## Test Your Understanding

1. **Conceptual:** If an autoencoder perfectly reconstructs the inputs, is it necessarily a good autoencoder? Why or why not?

2. **Implementation:** Modify the stacked autoencoder to have tied weights (decoder weights = encoder weights transposed). Does it train faster?

3. **Analysis:** Visualize which neurons in the VAE's latent space activate for different classes. What patterns do you see?

4. **Debugging:** The GAN we built shows mode collapse. Try these fixes:
   - Add label smoothing (use 0.9 for real, 0.1 for fake)
   - Train discriminator more often than generator
   - Add noise to discriminator inputs

5. **Extension:** Build a convolutional autoencoder for higher-quality reconstructions.

6. **Application:** Use an autoencoder for anomaly detection on a dataset of your choice (credit card fraud, network intrusions, etc.)

# 🚀 WHAT'S NEXT?

## Lecture 6B: Diffusion Models - Today's State of the Art

You've mastered:
- ✅ Autoencoders (compression, features, anomalies)
- ✅ VAEs (probabilistic generation)
- ✅ GANs (adversarial training)

Next, you'll learn:
- 🌊 **Diffusion Models** - Why GANs are being replaced
- 🎨 **Stable Diffusion** - Text-to-image generation
- ⚡ **Latent Diffusion** - Speed and quality together
- 🎯 **ControlNet** - Precise control over generation

**See you in Lecture 6B!** 🚀

---

<div style="text-align: center; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 10px;">
    <h1>🌟 From Compression to Creation 🌟</h1>
    <p style="font-size: 18px; font-style: italic;">"You started with pixels. Now you generate new realities."</p>
    <p><strong>SAIR Community - Sudanese Artificial Intelligence Research</strong></p>
    <p>🇸🇩 Building Sudan's AI Future, One Model at a Time</p>
</div>